In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import matplotlib.pyplot as plt

# Load featured datasets and models
amazon_featured = pd.read_csv("Amazon_featured.csv")
google_featured = pd.read_csv("GOOG_featured.csv")
netflix_featured = pd.read_csv("NFLX_featured.csv")

# Load trained models
amazon_model = tf.keras.models.load_model('amazon_lstm_model.h5')
google_model = tf.keras.models.load_model('google_lstm_model.h5')
netflix_model = tf.keras.models.load_model('netflix_lstm_model.h5')

# Function to prepare data for prediction (similar to training)
def prepare_prediction_data(df, target_col='Close', sequence_length=60):
    df = df.sort_values(by='Date').reset_index(drop=True)
    features = df.drop(columns=['Date', target_col])
    target = df[target_col]
    
    scaler_features = MinMaxScaler()
    features_scaled = scaler_features.fit_transform(features)
    
    scaler_target = MinMaxScaler()
    target_scaled = scaler_target.fit_transform(target.values.reshape(-1, 1))
    
    # Use the last sequence for prediction
    last_sequence = features_scaled[-sequence_length:]
    X_pred = np.array([last_sequence])
    
    return X_pred, scaler_target

# Function to predict next day price
def predict_next_day(model, df, name):
    X_pred, scaler_target = prepare_prediction_data(df)
    prediction_scaled = model.predict(X_pred)
    prediction = scaler_target.inverse_transform(prediction_scaled)[0][0]
    
    last_actual = df['Close'].iloc[-1]
    
    print(f"\n--- {name} Next Day Prediction ---")
    print(f"Last Actual Close: {last_actual:.4f}")
    print(f"Predicted Next Close: {prediction:.4f}")
    print(f"Change: {((prediction - last_actual) / last_actual * 100):.2f}%")
    
    return prediction, last_actual

# Predict for each stock
amazon_pred, amazon_last = predict_next_day(amazon_model, amazon_featured, "Amazon")
google_pred, google_last = predict_next_day(google_model, google_featured, "Google")
netflix_pred, netflix_last = predict_next_day(netflix_model, netflix_featured, "Netflix")

# Visualization function
def visualize_predictions(df, name, prediction, last_actual):
    plt.figure(figsize=(14, 7))
    
    # Plot historical close prices
    plt.plot(df['Date'], df['Close'], label='Historical Close Price', color='blue')
    
    # Add prediction point
    last_date = pd.to_datetime(df['Date'].iloc[-1])
    next_date = last_date + pd.Timedelta(days=1)
    plt.scatter(next_date, prediction, color='red', s=100, label='Predicted Next Day', zorder=5)
    
    plt.title(f'{name} Stock Price Prediction')
    plt.xlabel('Date')
    plt.ylabel('Close Price')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Visualize for each stock
visualize_predictions(amazon_featured, "Amazon", amazon_pred, amazon_last)
visualize_predictions(google_featured, "Google", google_pred, google_last)
visualize_predictions(netflix_featured, "Netflix", netflix_pred, netflix_last)

print("\nPrediction and visualization completed.")